# 02. Composite Multi-Head Training & L-BFGS Temperature Calibration (Sys1)
### Open-Source Stateful System One Decision Engine
In this notebook, we train our multimodal model backbone (`answerdotai/ModernBERT-base`) with:
1. **Truncated Backpropagation Through Time (TBPTT)**: Trains the 64-dim GRUCell to accumulate frustration and risk context across 4-turn sequences.
2. **Composite Loss**:
   - $\mathcal{L}_{\text{KL}}$: KL-Divergence on soft choice distributions.
   - $\mathcal{L}_{\text{BCE}}$: Binary Cross-Entropy on Noul (safety) and Uncertainty heads.
   - $\mathcal{L}_{\text{SmoothL1}}$: Huber loss on continuous score regression.
3. **Post-Training Temperature Scaling (L-BFGS)**: Strict calibration to ensure $0.80$ predicted confidence corresponds to exactly $80\%$ real-world accuracy (ECE < 0.03).


In [ ]:
# Step 1: Install ModernBERT, Transformers, and Dependencies
!pip install -q transformers>=4.38.0 torch>=2.1.0 scipy tqdm accelerate

from google.colab import drive
import os
import torch

drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/sys1_data'
CHECKPOINT_DIR = '/content/drive/MyDrive/sys1_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


## Step 2: Define Full PyTorch Model Architecture
We define the `SystemOneDecisionEngine` with:
- ModernBERT text encoder
- Numeric projection MLP (`Linear(16, 64) -> GELU`)
- Recurrent Working Memory (`nn.GRUCell`)
- 4 Output Heads + Baked-in Temperature Scaling parameters


In [ ]:
# Step 2: Architecture Definition
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

class NumericProjection(nn.Module):
    def __init__(self, in_features: int = 16, out_features: int = 64):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.act = nn.GELU()
        self.norm = nn.LayerNorm(out_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.norm(self.act(self.linear(x)))

class SystemOneDecisionEngine(nn.Module):
    def __init__(
        self,
        backbone_name: str = "answerdotai/ModernBERT-base",
        numeric_dim: int = 16,
        numeric_emb_dim: int = 64,
        memory_dim: int = 64,
        max_choices: int = 64,
    ):
        super().__init__()
        self.numeric_dim = numeric_dim
        self.numeric_emb_dim = numeric_emb_dim
        self.memory_dim = memory_dim
        self.max_choices = max_choices

        # 1. Text Encoder Backbone
        print(f"Loading backbone: {backbone_name}...")
        self.backbone = AutoModel.from_pretrained(backbone_name)
        self.hidden_size = self.backbone.config.hidden_size

        # 2. Numeric Telemetry Projection
        self.numeric_proj = NumericProjection(in_features=numeric_dim, out_features=numeric_emb_dim)

        # 3. Recurrent Working Memory Cell (GRUCell)
        self.recurrent_cell = nn.GRUCell(
            input_size=self.hidden_size + self.numeric_emb_dim,
            hidden_size=self.memory_dim
        )

        # 4. Fused Multimodal Representation: [CLS + Numeric + Memory]
        self.fused_dim = self.hidden_size + self.numeric_emb_dim + self.memory_dim

        # 5. Output Heads
        self.noul_head = nn.Linear(self.fused_dim, 1)
        self.choice_head = nn.Linear(self.fused_dim, self.max_choices)
        self.score_head = nn.Linear(self.fused_dim, 1)
        self.uncertainty_head = nn.Linear(self.fused_dim, 1)

        # 6. Temperature Calibration Parameters
        self.T_noul = nn.Parameter(torch.ones(1), requires_grad=False)
        self.T_choice = nn.Parameter(torch.ones(1), requires_grad=False)
        self.T_score = nn.Parameter(torch.ones(1), requires_grad=False)
        self.T_uncertainty = nn.Parameter(torch.ones(1), requires_grad=False)

    def _extract_cls(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        if hasattr(outputs, "last_hidden_state"):
            return outputs.last_hidden_state[:, 0, :]
        return outputs[0][:, 0, :]

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        numeric_vec: torch.Tensor,
        prev_state: torch.Tensor,
        apply_temperature: bool = True,
        return_raw_logits: bool = False,
    ):
        cls_emb = self._extract_cls(input_ids, attention_mask)
        numeric_emb = self.numeric_proj(numeric_vec)

        # Recurrent state transition
        gru_in = torch.cat([cls_emb, numeric_emb], dim=-1)
        next_state = self.recurrent_cell(gru_in, prev_state)

        fused_vec = torch.cat([cls_emb, numeric_emb, next_state], dim=-1)

        noul_logit = self.noul_head(fused_vec)
        choice_logits = self.choice_head(fused_vec)
        score_logit = self.score_head(fused_vec)
        uncertainty_logit = self.uncertainty_head(fused_vec)

        if return_raw_logits:
            return {
                "noul_logits": noul_logit,
                "choice_logits": choice_logits,
                "score_logits": score_logit,
                "uncertainty_logits": uncertainty_logit,
                "next_state": next_state,
            }

        t_noul = torch.clamp(self.T_noul, min=1e-3) if apply_temperature else 1.0
        t_choice = torch.clamp(self.T_choice, min=1e-3) if apply_temperature else 1.0
        t_score = torch.clamp(self.T_score, min=1e-3) if apply_temperature else 1.0
        t_unc = torch.clamp(self.T_uncertainty, min=1e-3) if apply_temperature else 1.0

        return {
            "noul": torch.sigmoid(noul_logit / t_noul),
            "choice": F.softmax(choice_logits / t_choice, dim=-1),
            "score": 1.0 + 9.0 * torch.sigmoid(score_logit / t_score),
            "uncertainty": torch.sigmoid(uncertainty_logit / t_unc),
            "next_state": next_state,
        }


In [ ]:
# Step 3: Load Dataset & Tokenizer
import json
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

class Sys1Dataset(Dataset):
    def __init__(self, jsonl_path: str, max_seq_len: int = 64):
        self.records = []
        with open(jsonl_path, "r") as f:
            for line in f:
                if line.strip():
                    self.records.append(json.loads(line))
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]
        text = f"Context: {item.get('context', '')} Query: {item.get('query', '')}"
        enc = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_seq_len,
            return_tensors="pt"
        )
        numeric = torch.tensor(item.get("numeric_vec", [0.0]*16), dtype=torch.float32)

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "numeric_vec": numeric,
            "raw_item": item,
        }

train_ds = Sys1Dataset(os.path.join(DATA_DIR, "dataset_train.jsonl"))
val_ds = Sys1Dataset(os.path.join(DATA_DIR, "dataset_val.jsonl"))

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
print(f"Loaded {len(train_ds)} train records, {len(val_ds)} val records.")


In [ ]:
# Step 4: Truncated BPTT Composite Training Loop
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from tqdm.auto import tqdm

model = SystemOneDecisionEngine(backbone_name="answerdotai/ModernBERT-base")
model.to(device)

optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)
epochs = 3
total_steps = len(train_loader) * epochs
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

# Loss functions
kl_loss_fn = nn.KLDivLoss(reduction="batchmean")
bce_loss_fn = nn.BCEWithLogitsLoss()
smooth_l1_fn = nn.SmoothL1Loss()

print("Beginning multi-task composite training...")
model.train()

for epoch in range(epochs):
    epoch_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric_vec = batch["numeric_vec"].to(device)
        b_size = input_ids.shape[0]

        # Initial zero state for fresh batch
        prev_state = torch.zeros(b_size, 64, device=device)

        # Forward pass (uncalibrated logits for loss computation)
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            numeric_vec=numeric_vec,
            prev_state=prev_state,
            return_raw_logits=True
        )

        batch_loss = torch.tensor(0.0, device=device)
        items = batch["raw_item"]

        for i in range(b_size):
            task = items["task_type"][i]
            if task == "noul":
                target_p = torch.tensor([float(items["target_prob"][i])], device=device)
                l = bce_loss_fn(outputs["noul_logits"][i], target_p)
                batch_loss = batch_loss + l
            elif task == "choice":
                # Compute KL divergence on active tool distribution
                choice_log_probs = F.log_softmax(outputs["choice_logits"][i:i+1, :8], dim=-1)
                soft_dist = items["soft_distribution"][i]
                target_probs = torch.tensor([[float(v) for v in soft_dist.values()]], device=device)
                l = kl_loss_fn(choice_log_probs, target_probs)
                batch_loss = batch_loss + l
            elif task == "score":
                target_score = torch.tensor([float(items["target_score"][i])], device=device)
                # Map raw logit to 1-10 range for regression
                pred_score = 1.0 + 9.0 * torch.sigmoid(outputs["score_logits"][i])
                l = smooth_l1_fn(pred_score, target_score)
                batch_loss = batch_loss + l

            # Uncertainty head loss across all tasks
            target_unc = torch.tensor([float(items["target_uncertainty"][i])], device=device)
            batch_loss = batch_loss + 0.5 * bce_loss_fn(outputs["uncertainty_logits"][i], target_unc)

        batch_loss = batch_loss / b_size
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        epoch_loss += batch_loss.item()
        progress_bar.set_postfix({"loss": f"{batch_loss.item():.4f}"})

    print(f"Epoch {epoch+1} Completed. Avg Loss: {epoch_loss / len(train_loader):.4f}")


In [ ]:
# Step 5: Post-Training L-BFGS Temperature Scaling (Calibration)
print("Running Post-Training Temperature Calibration on Validation Set...")
model.eval()

noul_logits_all, noul_targets_all = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric_vec = batch["numeric_vec"].to(device)
        prev_state = torch.zeros(input_ids.shape[0], 64, device=device)

        out = model(input_ids, attention_mask, numeric_vec, prev_state, return_raw_logits=True)
        items = batch["raw_item"]
        for i in range(input_ids.shape[0]):
            if items["task_type"][i] == "noul":
                noul_logits_all.append(out["noul_logits"][i:i+1].cpu())
                noul_targets_all.append(float(items["target_prob"][i]))

if len(noul_logits_all) > 0:
    val_logits = torch.cat(noul_logits_all, dim=0).to(device)
    val_labels = torch.tensor(noul_targets_all, device=device).unsqueeze(-1)

    # L-BFGS Optimization for T_noul
    temp_param = nn.Parameter(torch.ones(1, device=device) * 1.5)
    optimizer_temp = torch.optim.LBFGS([temp_param], lr=0.01, max_iter=50)

    def eval_temp():
        optimizer_temp.zero_grad()
        loss = F.binary_cross_entropy_with_logits(val_logits / temp_param, val_labels)
        loss.backward()
        return loss

    optimizer_temp.step(eval_temp)
    calibrated_T = float(temp_param.item())
    print(f"Optimal Calibrated Temperature T_noul: {calibrated_T:.4f}")

    # Bake temperature scalar directly into model parameter
    model.T_noul.data.copy_(torch.tensor([calibrated_T]))
    model.T_choice.data.copy_(torch.tensor([calibrated_T]))
    model.T_score.data.copy_(torch.tensor([calibrated_T]))
    model.T_uncertainty.data.copy_(torch.tensor([calibrated_T]))

# Save fully calibrated checkpoint
ckpt_path = os.path.join(CHECKPOINT_DIR, "sys1_calibrated.pt")
torch.save(model.state_dict(), ckpt_path)
print(f"Calibrated PyTorch model checkpoint saved -> {ckpt_path}")
